# 1D Damped Wave Equation — Dirichlet BCs

## Problem

$$\frac{\partial^2 u}{\partial t^2} + 2\gamma\,\frac{\partial u}{\partial t}
- \frac{\partial^2 u}{\partial x^2} = 0,
\qquad (t,x) \in (0,2) \times (0,1), \qquad \gamma = 0.2$$

**Initial conditions** (two, since the equation is second order in time)

$$u(0,x) = \sin(\pi x), \qquad
\frac{\partial u}{\partial t}\bigg|_{t=0} = -\gamma \sin(\pi x)$$

**Boundary conditions**

$$u(t,0) = 0, \qquad u(t,1) = 0$$

Hyperbolic. A string fixed at both ends, released from a sine displacement,
oscillating against a damping force proportional to velocity. The $2\gamma\,u_t$
term is the PDE analogue of the damped oscillator from Lecture 2.

## Exact solution

$$u_{\text{exact}}(t,x) = e^{-\gamma t}\cos(\omega t)\,\sin(\pi x),
\qquad \omega = \sqrt{\pi^2 - \gamma^2} \approx 3.1352$$

Separable: a fixed spatial shape $\sin(\pi x)$ whose amplitude oscillates at
frequency $\omega$ inside a decaying envelope $e^{-\gamma t}$. Damping both
shrinks the amplitude and shifts the frequency down from $\pi$.

## Method

$$u_\theta(t,x) = e^{-\gamma t}\sin(\pi x)
\;+\; t^2\,\sin(\pi x)\,\mathrm{NN}(\theta; t, x)$$

Verification of all four conditions:

| Condition | Required | Ansatz gives |
|---|---|---|
| $u(0,x)$ | $\sin(\pi x)$ | $\sin(\pi x)$ &nbsp;(second term has $t^2$) |
| $\partial_t u\|_{t=0}$ | $-\gamma\sin(\pi x)$ | $-\gamma\sin(\pi x)$ &nbsp;($t^2$ kills its own derivative too) |
| $u(t,0)$ | $0$ | $0$ &nbsp;($\sin 0 = 0$ in both terms) |
| $u(t,1)$ | $0$ | $0$ &nbsp;($\sin \pi = 0$ in both terms) |

The factor $t^2$ is required rather than $t$: it vanishes at $t=0$ **and** so
does its first derivative, which is what protects both initial conditions.

**Residual**

$$R(t,x) = \frac{\partial^2 u_\theta}{\partial t^2}
+ 2\gamma\,\frac{\partial u_\theta}{\partial t}
- \frac{\partial^2 u_\theta}{\partial x^2}$$

**Loss**

$$\mathcal{L}(\theta) = \frac{1}{NM}\sum_{i=1}^{N}\sum_{j=1}^{M} R(t_i,x_j)^2$$

## Note on the ansatz

Two informed choices were made here:

1. The lift $e^{-\gamma t}\sin(\pi x)$ rather than the minimal Taylor form
   $(1-\gamma t)\sin(\pi x)$ — it satisfies both initial conditions and already
   carries the correct decay envelope.
2. The vanishing factor $\sin(\pi x)$ rather than $x(1-x)$.

Choice 2 makes the whole trial function proportional to $\sin(\pi x)$, i.e.
separable with the correct spatial profile built in. Both are valid, but they
encode knowledge of the solution's structure. Section [N] compares against the
neutral factor $x(1-x)$.

In [16]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

GAMMA = 0.2
OMEGA = np.sqrt(np.pi**2 - GAMMA**2)
T_max, X_max = 2.0, 1.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("using", device)

using cuda


In [17]:
def grad(y,x):
    return torch.autograd.grad(
        y,x, grad_outputs = torch.ones_like(y), create_graph = True
    )[0]

In [18]:
class WavePINN(nn.Module):
    def __init__(self, width = 64, depth=4):
        super().__init__()
        layers = [nn.Linear(2,width),nn.Tanh()]
        for _ in range(depth-1):
            layers += [nn.Linear(width,width), nn.Tanh()]
        layers += [nn.Linear(width,1)]
        self.net = nn.Sequential(*layers)

    def forward(self,t,x):
        
        nn_out = self.net(torch.cat([t,x],dim=1))

        term1 = torch.exp(GAMMA * -t) * torch.sin(torch.pi * x)
        term2 = (t ** 2) * torch.sin(torch.pi * x) * nn_out

        return term1 + term2

In [19]:
#collocation points
N,M = 256,128
t_lin = torch.linspace(0, T_max, N)
x_lin = torch.linspace(0, X_max, M)
T_grid,X_grid = torch.meshgrid(t_lin, x_lin, indexing='ij')

t_col = T_grid.reshape(-1, 1).to(device).requires_grad_(True)
x_col = X_grid.reshape(-1, 1).to(device).requires_grad_(True)

print(t_col.shape)

torch.Size([32768, 1])


In [20]:
model = WavePINN(width = 64, depth = 4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def compute_loss():
    u = model(t_col,x_col)
    u_t = grad(u, t_col)
    u_tt = grad(u_t, t_col)
    u_x = grad(u, x_col)
    u_xx = grad(u_x, x_col)

    residual = u_tt + 2*GAMMA*u_t - u_xx

    return (residual**2).mean()


In [21]:
import time 

torch.cuda.synchronize()
t0 = time.time()
losses = []

for epoch in range(3000):
    optimizer.zero_grad()
    loss = compute_loss()
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:                                           # ← changed
        losses.append(loss.item()) 
    if epoch % 300 == 0:
        print(f"epoch {epoch:5d}  loss {loss.item():.3e}")

torch.cuda.synchronize() 
pinn_time = time.time()-t0
print(f"trained in {pinn_time:.1f}s")

epoch     0  loss 1.784e+01
epoch   300  loss 2.858e-01
epoch   600  loss 1.011e-01
epoch   900  loss 5.763e-02
epoch  1200  loss 2.914e-02
epoch  1500  loss 3.995e-02
epoch  1800  loss 1.414e-02
epoch  2100  loss 1.098e-02
epoch  2400  loss 8.805e-03
epoch  2700  loss 1.362e-02
trained in 53.6s


In [22]:
with torch.no_grad():
    tt = torch.linspace(0, T_max, 200).reshape(-1, 1)
    xx = torch.linspace(0, X_max, 100).reshape(-1, 1)
    TT, XX = torch.meshgrid(tt.flatten(), xx.flatten(), indexing='ij')
    t_ev = TT.reshape(-1, 1).to(device)                           # ← added .to(device)
    x_ev = XX.reshape(-1, 1).to(device)                           # ← added .to(device)
    u_pred = model(t_ev, x_ev).reshape(200, 100).cpu().numpy()    # ← added .cpu()

u_exact = (np.exp(-GAMMA * TT.numpy())                            # ← all three
           * np.cos(OMEGA * TT.numpy())                           #    lines are
           * np.sin(np.pi * XX.numpy()))                          #    new

pinn_err = np.abs(u_pred - u_exact).max()
print("PINN max error:", pinn_err)

PINN max error: 0.004610885985543461
